In [1]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!rm -rf colab_llm_utils

In [3]:
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1025, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 1025 (delta 67), reused 76 (delta 32), pack-reused 912 (from 2)
Receiving objects: 100% (1025/1025), 175.52 KiB | 7.63 MiB/s, done.
Resolving deltas: 100% (621/621), done.


In [4]:
!pip install openai pandas evaluate tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [5]:
import colab_llm_utils
from colab_llm_utils import configs

In [6]:
import pandas as pd
import os
import json
import time
from openai import OpenAI, OpenAIError
from google.colab import userdata
from tqdm.notebook import tqdm

In [7]:

try:
    api_key = userdata.get('OPENAI_TOKEN')
    client = OpenAI(api_key=api_key)
    print("✅ API Key loaded successfully.")
except Exception as e:
    print("❌ Error loading API key. Did you add 'OPENAI_API_KEY' to Colab Secrets?")


✅ API Key loaded successfully.


In [8]:
dataset_config = configs.datasets.triviaqa
model_config = configs.models.qwen3_8b

short_name = model_config['model_name'].split('/')[-1]

FEW_SHOT = True
P_TRUE = False
FACTUALITY_ONLY = True
COMPARISON = False

special_tag = ""
special_tag="thinking"

drive_path = f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/"

In [9]:
df_res = pd.read_pickle(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}.pickle"))

In [10]:
df_res

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,likes,all_probas,top_probas,chow_av,chow_sum,chow_quantile,log_chow_av,thinking_text,rouge,em,f1
0,Provide a short answer without explanation.\n ...,"[<think>\nOkay, let's tackle this question. Th...","{'aliases': ['(Harry) Sinclair Lewis', 'Harry ...","[{151667: 59.58333206176758, 1: -inf, 3: -inf,...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[59.58333206176758, 61.111106872558594, 54.166...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 0.39731527098905767, 0.93...",0.920517,276.154971,1.0,-0.139875,None,0.017621,0.0,0.019608
1,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking where Dame ...","{'aliases': ['Park Grove (1895)', 'York UA', '...","[{151667: 58.75, 1: -inf, 3: -inf, 6: -inf, 7:...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[58.75, 62.5, 55.416664123535156, 68.75, 47.5,...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",0.929969,183.203857,1.0,-0.096661,None,0.013605,0.0,0.014493
2,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking about when ...","{'aliases': ['30's', '30’s', '30s', '30s AD', ...","[{151667: 57.5, 1: -inf, 3: -inf, 6: -inf, 7: ...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[57.5, 61.45833206176758, 60.416664123535156, ...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 0.9579122720843811, 1.0, ...",0.855151,336.929369,1.0,-0.254022,None,0.007843,0.0,0.009050
3,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking about which...","{'aliases': ['Portogało', 'Republic of Portuga...","[{151667: 58.75, 1: -inf, 3: -inf, 6: -inf, 7:...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[58.75, 61.45833206176758, 53.33333206176758, ...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.948...",0.873287,231.421110,1.0,-0.208003,None,0.010695,0.0,0.011429
4,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking which city ...","{'aliases': ['Chi-Beria', 'Sayre language acad...","[{151667: 59.166664123535156, 1: -inf, 3: -inf...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[59.166664123535156, 62.15277099609375, 62.916...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",0.929864,209.219506,1.0,-0.119126,None,0.010989,0.0,0.011765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1895,Provide a short answer without explanation.\n ...,"[<think>\nOkay, let's see. The question is ask...","{'aliases': ['Hormone', 'Hormonal', 'Life horm...","[{151667: 57.916664123535156, 1: -inf, 3: -inf...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[57.916664123535156, 62.5, 61.25, 67.5, 50.0, ...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 0.6513548646660542, 0.937...",0.906856,274.777419,1.0,-0.140662,None,0.008621,0.0,0.009390
1896,Provide a short answer without explanation.\n ...,"[<think>\nOkay, let's see. The question is ask...","{'aliases': ['Pulmonary surgical procedures', ...","[{151667: 59.58333206176758, 1: -inf, 3: -inf,...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[59.58333206176758, 61.45833206176758, 56.6666...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 0.9375165132369381, 1.0, ...",0.950859,228.206163,1.0,-0.075076,None,0.022222,0.0,0.039216
1897,Provide a short answer without explanation.\n ...,"[<think>\nOkay, let's see. The question is ask...","{'aliases': ['Mutator genotype', 'Genotypical'...","[{151667: 57.916664123535156, 1: -inf, 3: -inf...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[57.916664123535156, 61.8055534362793, 58.75, ...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0

In [11]:

results = []
save_frequency = 100 # Save to CSV every 100 rows
output_filename = "scored_results_partial.csv"

print(f"Starting scoring for {len(df_res)} items...")

# Iterate through the DataFrame with a progress bar
for index, row in tqdm(df_res.iterrows(), total=len(df_res)):

    # Run the scorer
    response = row['responses']
    if isinstance(response, list):
        response = response[0]

    if dataset_config['task_type'] == 'qa':
        if FEW_SHOT:
            prompt = row['prompts'].split("Provide a short answer without explanation")[-1]
            prompt = prompt.split("\nTranslation:")[0]
        else:
          prompt = row['prompts']
        result = colab_llm_utils.api_scorer.score_qa_pair(client, row['prompts'], response, row['ans'])
        # Save data
        results.append({
            'index': index,
            'question': row['prompts'],
            'gpt_score': result.get('score'),
        })
    elif dataset_config['task_type'] == 'translation':
        if FEW_SHOT:
            prompt = row['prompts'].split("Translate  the following from ")[-1]
            prompt = prompt.split("\nTranslation:")[0]
        else:
          prompt = row['prompts']
        result = colab_llm_utils.api_scorer.score_translation(client, prompt, response, row['ans'])
        results.append({
            'index': index,
            'question': row['prompts'],
            'gpt_score': result.get('binary_adequacy'),
            'gpt_score_cont': result.get('adequacy'),
        })
    elif dataset_config['task_type'] == 'summarization':
        if COMPARISON:
            response2 = row['responses_large']
            if isinstance(response2, list):
                response2 = response2[0]
            prompt = row['prompts'].split(dataset_config['doc_type']+":")[-1]
            prompt = prompt.split("\n\nWrite a")[0]
            result = colab_llm_utils.api_scorer.compare_short_summary(client, prompt, response, response2, limit=dataset_config['limit'], crop=dataset_config['do_crop'], doc_type=dataset_config['doc_type'])
            results.append({
              'index': index,
              'question': prompt,
              'gpt_score': result.get('prefer_1'),
              'gpt_score_cont': result.get('prefer_2'),
          })
        else:
          if FEW_SHOT:
              prompt = row['prompts'].split(dataset_config['doc_type']+":")[-1]
              prompt = prompt.split("\n\nWrite a")[0]
          else:
            prompt = row['prompts']
          if dataset_config['do_crop']:
            result = colab_llm_utils.api_scorer.score_short_summary(client, prompt, response, limit=dataset_config['limit'], crop=dataset_config['do_crop'], doc_type=dataset_config['doc_type'])
          else:
            result = colab_llm_utils.api_scorer.score_summary(client, prompt, response)
          results.append({
              'index': index,
              'question': prompt,
              'gpt_score': result.get('faithful') and result.get('complete'),
              'gpt_score_cont': result.get('score'),
          })

    # Optional: Small sleep to be nice to the rate limits (though 4o-mini is fast)
    time.sleep(0.1)

    # Intermediate Save (so you don't lose progress)

    if (index + 1) % save_frequency == 0:
        int_df = pd.DataFrame(results)
        print(f"Mean score {int_df['gpt_score'].mean()}")
        int_df.to_csv(output_filename, index=False)



Starting scoring for 1900 items...


  0%|          | 0/1900 [00:00<?, ?it/s]

Mean score 0.48
Mean score 0.495
Mean score 0.5
Mean score 0.4875
Mean score 0.484
Mean score 0.46
Mean score 0.4614285714285714
Mean score 0.45375
Mean score 0.4444444444444444
Mean score 0.445
Mean score 0.44181818181818183
Mean score 0.4508333333333333
Mean score 0.4553846153846154
Mean score 0.4614285714285714
Mean score 0.4653333333333333
Mean score 0.479375
Mean score 0.49117647058823527
Mean score 0.51
Mean score 0.5310526315789473


In [12]:
df_res.head()

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,likes,all_probas,top_probas,chow_av,chow_sum,chow_quantile,log_chow_av,thinking_text,rouge,em,f1
0,Provide a short answer without explanation.\n ...,"[<think>\nOkay, let's tackle this question. Th...","{'aliases': ['(Harry) Sinclair Lewis', 'Harry ...","[{151667: 59.58333206176758, 1: -inf, 3: -inf,...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[59.58333206176758, 61.111106872558594, 54.166...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 0.39731527098905767, 0.93...",0.920517,276.154971,1.0,-0.139875,None,0.017621,0.0,0.019608
1,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking where Dame ...","{'aliases': ['Park Grove (1895)', 'York UA', '...","[{151667: 58.75, 1: -inf, 3: -inf, 6: -inf, 7:...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[58.75, 62.5, 55.416664123535156, 68.75, 47.5,...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",0.929969,183.203857,1.0,-0.096661,None,0.013605,0.0,0.014493
2,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking about when ...","{'aliases': ['30's', '30’s', '30s', '30s AD', ...","[{151667: 57.5, 1: -inf, 3: -inf, 6: -inf, 7: ...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[57.5, 61.45833206176758, 60.416664123535156, ...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 0.9579122720843811, 1.0, ...",0.855151,336.929369,1.0,-0.254022,None,0.007843,0.0,0.009050
3,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking about which...","{'aliases': ['Portogało', 'Republic of Portuga...","[{151667: 58.75, 1: -inf, 3: -inf, 6: -inf, 7:...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[58.75, 61.45833206176758, 53.33333206176758, ...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.948...",0.873287,231.421110,1.0,-0.208003,None,0.010695,0.0,0.011429
4,Provide a short answer without explanation.\n ...,"[<think>\nOkay, the user is asking which city ...","{'aliases': ['Chi-Beria', 'Sayre language acad...","[{151667: 59.166664123535156, 1: -inf, 3: -inf...","[151644, 872, 198, 60424, 264, 2805, 4226, 204...",{},{},"[59.166664123535156, 62.15277099609375, 62.916...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",0.929864,209.219506,1.0,-0.119126,None,0.010989,0.0,0.011765


In [13]:
# Final Save
final_df = pd.DataFrame(results)
df_res['gpt_score'] = final_df['gpt_score']
#df_res['gpt_score_alt'] = final_df['gpt_score_alt']
if dataset_config['task_type'] == 'translation' or dataset_config['task_type'] == "summarization":
  df_res['gpt_score_cont'] = final_df['gpt_score_cont']

In [14]:
final_df['gpt_score'].mean()

np.float64(0.5310526315789473)

In [15]:
final_df.to_csv(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}scored_results_final.csv"), index=False)

print("✅ Scoring Complete!")
# Display first few rows
final_df.head()

✅ Scoring Complete!


,index,question,gpt_score
0,0,Provide a short answer without explanation.\n ...,1
1,1,Provide a short answer without explanation.\n ...,0
2,2,Provide a short answer without explanation.\n ...,1
3,3,Provide a short answer without explanation.\n ...,1
4,4,Provide a short answer without explanation.\n ...,0


In [16]:
df_res.to_pickle(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}.pickle"))

In [17]:
pd.read_csv(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}scored_results_final.csv"))


,index,question,gpt_score
0,0,Provide a short answer without explanation.\n ...,1
1,1,Provide a short answer without explanation.\n ...,0
2,2,Provide a short answer without explanation.\n ...,1
3,3,Provide a short answer without explanation.\n ...,1
4,4,Provide a short answer without explanation.\n ...,0
...,...,...,...
1895,1895,Provide a short answer without explanation.\n ...,1
1896,1896,Provide a short answer without explanation.\n ...,1
1897,1897,Provide a short answer without explanation.\n ...,1
1898,1898,Provide a short answer without explanation.\n ...,1


In [18]:
df

NameError: name 'df' is not defined